In [16]:
# ==========================================
# ШАГ 1: ИМПОРТ И ЗАГРУЗКА ДАННЫХ
# ==========================================
import pandas as pd
import numpy as np

# Загружаем исходный датасет
df = pd.read_csv('india_personal_loan_default_risk_2026.csv')

print("=== ОБЩАЯ ИНФОРМАЦИЯ ===\n")
print(f"Количество строк: {len(df)}")
print(f"Количество колонок: {len(df.columns)}")
print(f"\nНазвания колонок:\n{df.columns.tolist()}")
print("\n=== ПЕРВЫЕ 3 СТРОКИ ===\n")
print(df.head(3))
print("\n=== ТИПЫ ДАННЫХ И ПРОПУСКИ ===\n")
print(df.dtypes)
print("\nКоличество пропусков в каждой колонке:")
print(df.isnull().sum())
print("\n=== ОСНОВНАЯ СТАТИСТИКА ПО ЧИСЛОВЫМ ПЕРЕМЕННЫМ ===\n")
print(df.describe())

print("\n=== СТАТИСТИКА ПО КАТЕГОРИАЛЬНЫМ ПЕРЕМЕННЫМ ===\n")
categorical_cols = ['gender', 'marital_status', 'city_tier', 'state', 'employment_type', 
                    'loan_purpose', 'collateral_provided']
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().head())


# ==========================================
# ШАГ 2: СОЗДАЕМ НОВЫЕ ПРИЗНАКИ (FEATURE ENGINEERING)
# ==========================================
print("\n=== СОЗДАЕМ НОВЫЕ ПРИЗНАКИ ===\n")

# 1. Отношение ежемесячного платежа к доходу (насколько кредит "тяжелый")
df['emi_to_income_ratio'] = df['existing_emi_monthly_inr'] / df['monthly_income_inr']

# 2. Отношение запрашиваемой суммы к годовому доходу
df['loan_to_annual_income'] = df['loan_amount_requested_inr'] / (df['monthly_income_inr'] * 12)

# 3. Возрастные группы (для анализа в Tableau)
df['age_group'] = pd.cut(df['age'], 
                         bins=[0, 25, 35, 45, 55, 100], 
                         labels=['18-25', '26-35', '36-45', '46-55', '56+'])

# 4. Группы по кредитному скору
df['cibil_group'] = pd.cut(df['cibil_score'], 
                           bins=[0, 600, 700, 800, 900], 
                           labels=['Низкий (<600)', 'Средний (600-700)', 'Хороший (700-800)', 'Отличный (800+)'])

# 5. Группы по доходу
df['income_group'] = pd.cut(df['monthly_income_inr'], 
                            bins=[0, 25000, 50000, 100000, 1000000], 
                            labels=['До 25k', '25k-50k', '50k-100k', '100k+'])

# 6. Риск дефолта (бинарный, для удобства)
df['default_risk'] = df['default_flag'].map({0: 'Выплатил', 1: 'Дефолт'})

print("Новые колонки созданы:", df.columns.tolist()[-6:])


# ==========================================
# ШАГ 3: КЛЮЧЕВЫЕ МЕТРИКИ И СРАВНЕНИЕ ГРУПП
# ==========================================
print("\n=== КЛЮЧЕВЫЕ МЕТРИКИ ДЛЯ ДАШБОРДА ===\n")

# Общая статистика
total_loans = len(df)
total_defaults = df['default_flag'].sum()
default_rate = (total_defaults / total_loans) * 100

print(f"Всего заявок: {total_loans:,}")
print(f"Дефолтов: {total_defaults:,}")
print(f"Доля дефолтов: {default_rate:.2f}%")
print(f"Средний кредитный скор: {df['cibil_score'].mean():.0f}")
print(f"Средний доход: {df['monthly_income_inr'].mean():.0f} INR")
print(f"Средняя сумма кредита: {df['loan_amount_requested_inr'].mean():.0f} INR")

# Статистика по дефолтникам и не-дефолтникам
print("\n=== СРАВНЕНИЕ ДЕФОЛТНИКОВ И НЕ-ДЕФОЛТНИКОВ ===\n")
grouped = df.groupby('default_flag').agg({
    'cibil_score': ['mean', 'min', 'max'],
    'monthly_income_inr': ['mean', 'min', 'max'],
    'loan_amount_requested_inr': ['mean', 'min', 'max'],
    'existing_loans_count': ['mean', 'max'],
    'late_payments_last_12m': ['mean', 'max']
})
print(grouped)


# ==========================================
# ШАГ 4: КРАТКАЯ СВОДКА (ВЫВОДЫ ДЛЯ РАБОТОДАТЕЛЯ)
# ==========================================
print("\n=== КРАТКАЯ СВОДКА ===\n")

# 1. Считаем фактические цифры из датасета
low_cibil_defaults = df[df['cibil_group'] == 'Низкий (<600)']['default_flag'].mean() * 100
high_cibil_defaults = df[df['cibil_group'] == 'Хороший (700-800)']['default_flag'].mean() * 100
business_defaults = df[df['employment_type'] == 'Business Owner']['default_flag'].mean() * 100
wedding_defaults = df[df['loan_purpose'] == 'Wedding']['default_flag'].mean() * 100
debt_consolidation_defaults = df[df['loan_purpose'] == 'Debt Consolidation']['default_flag'].mean() * 100

# ДОБАВЛЯЕМ НОВЫЙ РАСЧЕТ ПРО ПРОСРОЧКИ
late_payers_defaults = df[df['late_payments_last_12m'] > 0]['default_flag'].mean() * 100
no_late_payers_defaults = df[df['late_payments_last_12m'] == 0]['default_flag'].mean() * 100

# 2. Выводим выводы
print("ТОП-5 ФАКТОРОВ РИСКА:")
print(f"1. Кредитный скор (CIBIL): {low_cibil_defaults:.1f}% дефолтов (против {high_cibil_defaults:.1f}% у 700-800)")
print(f"2. Просрочки: у клиентов с прошлыми просрочками дефолт {late_payers_defaults:.1f}% (против {no_late_payers_defaults:.1f}% у остальных)")
print(f"3. Доля платежа (EMI): Высокое соотношение EMI к доходу повышает риск")
print(f"4. Тип занятости: Владельцы бизнеса рискуют чаще ({business_defaults:.1f}% дефолтов)")
print(f"5. Цель кредита: Свадьбы ({wedding_defaults:.1f}%) и консолидация долгов ({debt_consolidation_defaults:.1f}%) — самые рискованные цели")

# Кто чаще всего берет кредиты?
print("\nКТО ЧАЩЕ БЕРЕТ КРЕДИТЫ?")
print(f"Больше всего заявок от: {df['employment_type'].mode()[0]}")
print(f"Самый популярный штат: {df['state'].mode()[0]}")
print(f"Самый популярный город: {df['city_tier'].mode()[0]}")
print(f"Самый популярный пол: {df['gender'].mode()[0]}")


# ==========================================
# ШАГ 5: ПОДГОТОВКА ДАННЫХ ДЛЯ TABLEAU
# ==========================================
print("\n=== ПОДГОТОВКА ДАННЫХ ДЛЯ TABLEAU ===\n")

# 1. Данные для графика "Дефолты по цели кредита"
purpose_stats = df.groupby('loan_purpose').agg({
    'default_flag': ['count', 'mean', 'sum']
}).reset_index()
purpose_stats.columns = ['loan_purpose', 'total_loans', 'default_rate', 'defaults_count']
purpose_stats['default_rate'] = purpose_stats['default_rate'] * 100
purpose_stats = purpose_stats.sort_values('default_rate', ascending=False)

print("=== ДЕФОЛТЫ ПО ЦЕЛИ КРЕДИТА ===")
print(purpose_stats.head(10))

# 2. Данные для графика "Дефолты по типу занятости"
employment_stats = df.groupby('employment_type').agg({
    'default_flag': ['count', 'mean', 'sum']
}).reset_index()
employment_stats.columns = ['employment_type', 'total_loans', 'default_rate', 'defaults_count']
employment_stats['default_rate'] = employment_stats['default_rate'] * 100
employment_stats = employment_stats.sort_values('default_rate', ascending=False)

print("\n=== ДЕФОЛТЫ ПО ТИПУ ЗАНЯТОСТИ ===")
print(employment_stats)

# 3. Данные для графика "Дефолты по возрастной группе"
age_stats = df.groupby('age_group').agg({
    'default_flag': ['count', 'mean', 'sum']
}).reset_index()
age_stats.columns = ['age_group', 'total_loans', 'default_rate', 'defaults_count']
age_stats['default_rate'] = age_stats['default_rate'] * 100

print("\n=== ДЕФОЛТЫ ПО ВОЗРАСТНОЙ ГРУППЕ ===")
print(age_stats)

# 4. Данные для графика "Дефолты по кредитному скору"
cibil_stats = df.groupby('cibil_group').agg({
    'default_flag': ['count', 'mean', 'sum']
}).reset_index()
cibil_stats.columns = ['cibil_group', 'total_loans', 'default_rate', 'defaults_count']
cibil_stats['default_rate'] = cibil_stats['default_rate'] * 100

print("\n=== ДЕФОЛТЫ ПО КРЕДИТНОМУ СКОРУ ===")
print(cibil_stats)


# ==========================================
# ШАГ 6: СОХРАНЯЕМ ДАННЫЕ
# ==========================================
print("\n=== СОХРАНЯЕМ ДАННЫЕ ДЛЯ TABLEAU ===\n")

# 1. Сохраняем полный датасет с новыми колонками
df.to_csv('loan_data_for_tableau.csv', index=False)
print("✓ Полные данные сохранены в 'loan_data_for_tableau.csv'")

# 2. Сохраняем агрегированные данные для быстрых графиков
with pd.ExcelWriter('preview_aggregated_data.xlsx') as writer:
    purpose_stats.to_excel(writer, sheet_name='По_цели_кредита', index=False)
    employment_stats.to_excel(writer, sheet_name='По_типу_занятости', index=False)
    age_stats.to_excel(writer, sheet_name='По_возрасту', index=False)
    cibil_stats.to_excel(writer, sheet_name='По_кредитному_скору', index=False)
    
    # Добавляем сводную таблицу для географического анализа (по штатам)
    state_stats = df.groupby('state').agg({
        'default_flag': ['count', 'mean', 'sum'],
        'monthly_income_inr': 'mean',
        'cibil_score': 'mean'
    }).reset_index()
    state_stats.columns = ['state', 'total_loans', 'default_rate', 'defaults_count', 
                           'avg_income', 'avg_cibil']
    state_stats['default_rate'] = state_stats['default_rate'] * 100
    state_stats = state_stats.sort_values('default_rate', ascending=False)
    state_stats.to_excel(writer, sheet_name='По_штатам', index=False)

print("✓ Агрегированные данные сохранены в 'preview_aggregated_data.xlsx'")

=== ОБЩАЯ ИНФОРМАЦИЯ ===

Количество строк: 25000
Количество колонок: 22

Названия колонок:
['applicant_id', 'age', 'gender', 'marital_status', 'city_tier', 'state', 'employment_type', 'bank_account_vintage_years', 'monthly_income_inr', 'existing_loans_count', 'existing_emi_monthly_inr', 'credit_utilization_ratio_pct', 'num_credit_inquiries_last_6m', 'late_payments_last_12m', 'loan_amount_requested_inr', 'loan_tenure_months', 'loan_purpose', 'collateral_provided', 'cibil_score', 'interest_rate_offered_pct', 'default_risk_score', 'default_flag']

=== ПЕРВЫЕ 3 СТРОКИ ===

  applicant_id  age  gender marital_status city_tier          state  \
0    APP106868   22    Male         Single    Tier 3        Haryana   
1    APP124016   35  Female       Divorced    Tier 2        Gujarat   
2    APP109668   41    Male        Married    Tier 2  Uttar Pradesh   

         employment_type  bank_account_vintage_years  monthly_income_inr  \
0         Salaried - PSU                         0.5          

C:\Users\Artem\AppData\Local\Temp\ipykernel_17856\454420488.py:152: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_stats = df.groupby('age_group').agg({
C:\Users\Artem\AppData\Local\Temp\ipykernel_17856\454420488.py:162: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cibil_stats = df.groupby('cibil_group').agg({


✓ Полные данные сохранены в 'loan_data_for_tableau.csv'
✓ Агрегированные данные сохранены в 'preview_aggregated_data.xlsx'
